In [ ]:
import sys
sys.path.append(".")
sys.path.append("./collisionChecker")

import matplotlib.pyplot as plt
import time
import importlib
%matplotlib inline

from shapely.geometry import Point, LineString
from shapely import plotting

# Benchmark Suite importieren (mit Reload falls schon geladen)
import IPTestSuiteBenchmark
import IPTestSuitePlanarBenchmark
importlib.reload(IPTestSuiteBenchmark)
importlib.reload(IPTestSuitePlanarBenchmark)
from IPTestSuiteBenchmark import benchList, getBenchmarks2DoF, getBenchmarks3DoF, printBenchmarkOverview

# Planer importieren
from IPLazyPRM import LazyPRM
from IPBasicPRM import BasicPRM
from IPVisibilityPRM import VisPRM
from IPVisibilityPRMRound import VisPRMRound
from IPRoundtripPlanner import RoundtripPlanner

#Animator importieren
from PathAnimator import PathAnimator

# Übersicht
printBenchmarkOverview()
print(f"\nAnzahl Benchmarks geladen: {len(benchList)}")
for i, b in enumerate(benchList):
    print(f"  {i+1}. {b.name}")

In [ ]:
PLANNERS = {
    "LazyPRM": {
        "config": {
            "initialRoadmapSize": 40,
            "updateRoadmapSize": 20,
            "kNearest": 5,
            "maxIterations": 40
        },
        "class": LazyPRM
    },
    "BasicPRM": {
        "config": {
            "radius": 5.0,             # Suchradius für Nachbarn
            "numNodes": 300,           # Anzahl der zu generierenden Knoten
            "useKDTree": True          # KDTree für schnelle Nachbarsuche (optional)
        },
        "class": BasicPRM
    },
    "VisPRM": {
        "config": {
            "ntry": 40                 # Anzahl Versuche für Roadmap-Erstellung
        },
        "class": VisPRM,
    },
    "VisPRMRound": {
        "config": {
            "ntry": 40                # Roundtrip-Variante nutzt eigene TSP-Logik
        },
        "class": VisPRMRound
    },
}

In [ ]:
# ============================================================================
# 0. SETUP: Patch für Kollisionszähler
# ============================================================================
from collisionChecker.KinChainCollisionChecker import KinChainCollisionChecker

if not hasattr(KinChainCollisionChecker, '_counting_patched'):
    _orig_pointInCollision = KinChainCollisionChecker.pointInCollision
    
    def pointInCollision_counted(self, pos):
        if not hasattr(self, 'collision_calls'):
            self.collision_calls = 0
        self.collision_calls += 1
        return _orig_pointInCollision(self, pos)
        
    KinChainCollisionChecker.pointInCollision = pointInCollision_counted
    KinChainCollisionChecker._counting_patched = True
    print("KinChainCollisionChecker: Kollisions-Zähler aktiviert.")
else:
    print("Patch bereits aktiv.")

In [ ]:
# ============================================================================
# 1. BERECHNUNG
# ============================================================================

from IPTestSuitePlanarBenchmark import benchList
import IPPlanarManipulator 
benchmarks_planar = [b for b in benchList if b.name in ["PlanarArm_2DoF", "PlanarArm_3DoF"]]
planner_names = list(PLANNERS.keys())
results_planar = []

for bench in benchmarks_planar:
    print(f"\nBenchmark: {bench.name}")
    print("-" * 50)
    env = bench.collisionChecker
    start_pos = bench.startList[0]
    goal_list = bench.goalList

    for planner_name in planner_names:
        try:
            planner_class = PLANNERS[planner_name]["class"]
            planner_config = PLANNERS[planner_name]["config"]
            base_planner = planner_class(env)

            before = getattr(env, "collision_calls", 0)
            start_time = time.time()
            if planner_name == "VisPRMRound":
                path = base_planner.planPath([start_pos], goal_list, planner_config)
            else:
                roundtrip = RoundtripPlanner(env, base_planner)
                path = roundtrip.planPath([start_pos], goal_list, planner_config)
            end_time = time.time()

            after = getattr(env, "collision_calls", 0)
            n_collisions = after - before

            roadmap_size = getattr(base_planner, "graph", None)
            if roadmap_size is not None and hasattr(roadmap_size, "size"):
                roadmap_size = roadmap_size.size()
            else:
                roadmap_size = None

            planning_time = end_time - start_time
            path_length = len(path)

            results_planar.append({
                "benchmark": bench.name,
                "planner": planner_name,
                "time": planning_time,
                "path_points": path_length,
                "path": path,
                "success": True,
                "collision_checks": n_collisions,
                "roadmap_size": roadmap_size
            })

            print(f"  {planner_name}: {planning_time:.3f}s, {path_length} Punkte, {n_collisions} CollisionChecks, Roadmap: {roadmap_size}")

        except Exception as e:
            results_planar.append({
                "benchmark": bench.name,
                "planner": planner_name,
                "time": None,
                "path_points": None,
                "path": None,
                "success": False,
                "collision_checks": None,
                "roadmap_size": None
            })
            print(f"  {planner_name}: Fehler - {str(e)[:50]}...")

print("\n" + "=" * 70)

In [ ]:
from shapely import plotting
from collisionChecker.KinChainCollisionChecker import KinChainCollisionChecker

# Wir definieren die Methode neu
def drawObstacles_patched(self, ax, inWorkspace=False):
    if inWorkspace:
        for key, value in self.scene.items():
            plotting.plot_polygon(value, add_points=False, color='red', ax=ax)

KinChainCollisionChecker.drawObstacles = drawObstacles_patched

print("Fix angewendet: KinChainCollisionChecker")

In [ ]:
# ============================================================================
# 2. ERGEBNISSE PLOTTEN: Planar Manipulator (Arm)
# ============================================================================
import numpy as np

# Nur erfolgreiche Ergebnisse filtern
successful_results_planar = [r for r in results_planar if r['success']]

# Anzahl Plots bestimmen
n_benchmarks_planar = len(benchmarks_planar)
n_planners_planar = len(planner_names)

if successful_results_planar and n_benchmarks_planar > 0:
    fig, axes = plt.subplots(n_benchmarks_planar, n_planners_planar, 
                             figsize=(5*n_planners_planar, 5*n_benchmarks_planar))
    
    # Sicherstellen, dass axes immer ein 2D-Array ist (auch bei 1x1 oder 1xN Plots)
    if n_benchmarks_planar == 1 and n_planners_planar == 1:
        axes = np.array([[axes]])
    elif n_benchmarks_planar == 1:
        axes = np.array([axes])
    elif n_planners_planar == 1:
        axes = np.array([axes]).reshape(-1, 1)

    for bench_idx, bench in enumerate(benchmarks_planar):
        for plan_idx, planner_name in enumerate(planner_names):
            ax = axes[bench_idx, plan_idx]
            
            # Das passende Ergebnis für diesen Benchmark und Planer suchen
            result = next((r for r in successful_results_planar 
                           if r['benchmark'] == bench.name and r['planner'] == planner_name), None)

            env = bench.collisionChecker
            
            # --- 1. Arbeitsraum-Limits setzen ---
            # WICHTIG: env.getEnvironmentLimits() liefert bei KinChain oft die Winkel-Limits (z.B. -3.14 bis 3.14).
            # Wir brauchen aber die Koordinaten für den Plot (z.B. 0 bis 30).
            # Da diese nicht direkt gespeichert sind, setzen wir hier feste Werte passend zur Szene (ca. 30x30).
            ax.set_xlim(0, 30)
            ax.set_ylim(0, 30)
            ax.grid(True, alpha=0.3)
            ax.set_aspect('equal')

            # --- 2. Hindernisse zeichnen ---
            # Bei KinChainCollisionChecker muss explizit inWorkspace=True gesetzt werden!
            env.drawObstacles(ax, inWorkspace=True)

            title = f"{bench.name}\n{planner_name}"

            if result and result['path']:
                path = result['path']
                title += f": {result['time']:.2f}s, {len(path)} Pkt."
                
                # Hilfsfunktion, um den Roboter in einer bestimmten Farbe zu zeichnen
                def plot_robot_chain(environment, configuration, ax_obj, color, alpha=1.0, linestyle='-'):
                    environment.kin_chain.move(configuration)
                    transforms = environment.kin_chain.get_transforms()
                    # Segmente zeichnen
                    xs = [t[0] for t in transforms]
                    ys = [t[1] for t in transforms]
                    ax_obj.plot(xs, ys, color=color, alpha=alpha, linewidth=2, linestyle=linestyle)
                    # Gelenke als Punkte
                    ax_obj.plot(xs, ys, 'o', color=color, alpha=alpha, markersize=4)

                # --- 3. Pfad visualisieren (Stroboskop-Effekt) ---
                # Wir zeichnen den Roboter an ca. 10-15 Zwischenpositionen
                step = max(1, len(path) // 15)
                
                # Endeffektor-Spur speichern
                ee_trace_x = []
                ee_trace_y = []

                for i in range(0, len(path), step):
                    # Zwischenschritte transparent blau
                    plot_robot_chain(env, path[i], ax, color='blue', alpha=0.15)
                    
                    # Endeffektor Position für Spur speichern
                    current_eff = env.kin_chain.get_transforms()[-1]
                    ee_trace_x.append(current_eff[0])
                    ee_trace_y.append(current_eff[1])

                # Letzten Punkt auch hinzufügen für Spur
                env.kin_chain.move(path[-1])
                last_eff = env.kin_chain.get_transforms()[-1]
                ee_trace_x.append(last_eff[0])
                ee_trace_y.append(last_eff[1])

                # Spur des Endeffektors zeichnen (gestrichelt)
                ax.plot(ee_trace_x, ee_trace_y, 'b--', linewidth=1, alpha=0.6, label='TCP Spur')

                # --- 4. Start und Ziele hervorheben ---
                
                # Start (Grün, voll deckend)
                plot_robot_chain(env, path[0], ax, color='green', alpha=1.0)
                
                # Alle Ziele aus der Benchmark-Definition (Rot)
                for goal_conf in bench.goalList:
                    plot_robot_chain(env, goal_conf, ax, color='red', alpha=0.6, linestyle='--')

            else:
                title += ": FEHLER / Kein Pfad"

            ax.set_title(title, fontsize=10)

    plt.tight_layout()
    plt.show()

    # Tabelle für Planar ausgeben
    print("\nERGEBNISTABELLE PLANAR:")
    print("-" * 100)
    print(f"{'Benchmark':<20} {'Planner':<15} {'Zeit (s)':<12} {'Punkte':<10} {'Roadmap':<10} {'CollChecks':<12}")
    print("-" * 100)
    for r in successful_results_planar:
        print(f"{r['benchmark']:<20} {r['planner']:<15} {r['time']:<12.3f} {r['path_points']:<10} {r['roadmap_size']:<10} {r['collision_checks']:<12}")

else:
    print("Keine erfolgreichen Planar-Ergebnisse zum Plotten vorhanden.")

In [ ]:
# ============================================================================
# 3. AUSWERTUNG: Plots & Animation (Korrigiert: Mit Zielen)
# ============================================================================
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Wir filtern nur die erfolgreichen Ergebnisse aus der Berechnung davor
successful_planar = [r for r in results_planar if r['success']]

if not successful_planar:
    print("Keine erfolgreichen Pfade für die Auswertung.")
else:
    # --- A) Balkendiagramme ---
    bench_names = sorted(list(set(r['benchmark'] for r in successful_planar)))
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    x = np.arange(len(bench_names))
    width = 0.15
    
    metrics = [
        ('time', 'Planungszeit (s)', axes[0, 0]),
        ('path_points', 'Pfadpunkte', axes[0, 1]),
        ('roadmap_size', 'Roadmap Size', axes[1, 0]),
        ('collision_checks', 'Collision Checks', axes[1, 1])
    ]

    for metric_key, label, ax in metrics:
        for i, planner in enumerate(planner_names):
            vals = []
            for b_name in bench_names:
                val = next((r[metric_key] for r in successful_planar 
                           if r['benchmark'] == b_name and r['planner'] == planner), 0)
                vals.append(val)
            ax.bar(x + i*width, vals, width, label=planner, alpha=0.8)
            
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.set_xticks(x + width * (len(planner_names)-1)/2)
        ax.set_xticklabels(bench_names, rotation=15)
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

    # --- B) Animationen ---
    print("\nGeneriere Animationen...")
    
    def interpolate_path(path, num_frames=50):
        if not path or len(path) < 2: return np.array(path if path else [])
        path = np.array(path)
        dists = np.linalg.norm(np.diff(path, axis=0), axis=1)
        total_dist = np.sum(dists)
        if total_dist == 0: return np.array([path[0]] * num_frames)
        cum_dist = np.r_[0, np.cumsum(dists)]
        target = np.linspace(0, total_dist, num_frames)
        res = np.zeros((num_frames, path.shape[1]))
        for i in range(path.shape[1]):
            res[:, i] = np.interp(target, cum_dist, path[:, i])
        return res

    for res in successful_planar:
        bench = next((b for b in benchList if b.name == res['benchmark']), None)
        if not bench: continue
        
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.set_xlim(0, 30); ax.set_ylim(0, 30); ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_title(f"{res['benchmark']}: {res['planner']}\n(Zeit: {res['time']:.2f}s | Punkte: {res['path_points']})", fontsize=10)
        
        # 1. Hindernisse zeichnen
        if hasattr(bench.collisionChecker, 'scene'):
            from shapely import plotting
            for val in bench.collisionChecker.scene.values():
                plotting.plot_polygon(val, ax=ax, add_points=False, color='red', alpha=0.4)
        
        # 2. Startposition zeichnen (Grün, statisch)
        bench.collisionChecker.kin_chain.move(bench.startList[0])
        ts = bench.collisionChecker.kin_chain.get_transforms()
        ax.plot([t[0] for t in ts], [t[1] for t in ts], 'o-', color='green', alpha=0.6, linewidth=2, label='Start')

        # 3. ALLE Ziele zeichnen (Rot, statisch) -> HIER WAR DER FEHLER
        for goal_conf in bench.goalList:
            bench.collisionChecker.kin_chain.move(goal_conf)
            ts = bench.collisionChecker.kin_chain.get_transforms()
            ax.plot([t[0] for t in ts], [t[1] for t in ts], 'o-', color='red', alpha=0.4, linewidth=2)
            
        # Dummy-Plot für Legende (damit "Ziel" nur einmal auftaucht)
        ax.plot([], [], 'o-', color='red', alpha=0.4, label='Ziel(e)')
        
        # 4. Pfad animieren
        anim_path = interpolate_path(res['path'], num_frames=50)
        line_arm, = ax.plot([], [], 'o-', lw=3, color='blue', label='Roboter')
        line_trace, = ax.plot([], [], '--', lw=1, color='orange', alpha=0.7) # Spur
        trace_x, trace_y = [], []
        
        ax.legend(loc='upper right', fontsize=8)

        def update(frame):
            # Roboter bewegen
            bench.collisionChecker.kin_chain.move(anim_path[frame])
            ts = bench.collisionChecker.kin_chain.get_transforms()
            xs = [t[0] for t in ts]
            ys = [t[1] for t in ts]
            
            # Arm zeichnen
            line_arm.set_data(xs, ys)
            
            # Spur zeichnen
            trace_x.append(xs[-1])
            trace_y.append(ys[-1])
            line_trace.set_data(trace_x, trace_y)
            return line_arm, line_trace

        try:
            ani = FuncAnimation(fig, update, frames=len(anim_path), interval=60, blit=True)
            display(HTML(ani.to_jshtml()))
        except Exception as e:
            print(f"Fehler bei Animation: {e}")
            
        plt.close(fig)

print("Fertig.")